# Probability Distributions in Practice
### From the definitions to two working case studies: LLM text completion and digital image processing

---

## Why one notebook for two very different applications

A language model choosing the next token and an image processing routine choosing a threshold are doing the
same thing: **putting a probability distribution over a finite set of outcomes and then acting on it.** The
vocabulary of an LLM is a categorical distribution over ~50,000 symbols; the histogram of a grayscale image is
a categorical distribution over 256 symbols. Temperature scaling in a decoder and histogram equalization in a
scanner driver are both transformations of a distribution. Once the shared machinery is clear, techniques
transfer between the two fields instead of being memorized separately.

## Learning objectives

1. State and compute PMFs, PDFs, CDFs, expectation, and variance, and verify them numerically.
2. Recognize which distribution family fits which physical or computational mechanism.
3. Use entropy, cross-entropy, KL divergence, and perplexity as measurement tools, not decoration.
4. Estimate distribution parameters from data (MLE and MAP) and know when each is appropriate.
5. **Case study 1:** implement and compare LLM decoding strategies (greedy, temperature, top-k, top-p) on a
   real, trained n-gram language model, and evaluate them with perplexity.
6. **Case study 2:** use distributions for histogram equalization, noise identification, Otsu thresholding,
   Bayesian pixel classification, and compression bounds.

## Roadmap

| Part | Content | Why it matters downstream |
|---|---|---|
| 1 | Setup and helper functions | reproducible experiments |
| 2 | Random variables, PMF, CDF, expectation | the vocabulary |
| 3 | Discrete families | token distributions, impulse noise, photon counts |
| 4 | Continuous families | sensor noise, pixel intensities, logits |
| 5 | Sampling and the inverse-CDF transform | token sampling **and** histogram equalization |
| 6 | Entropy, cross-entropy, KL, perplexity | training loss and image information content |
| 7 | Estimation: MLE and MAP | fitting distributions to real data |
| 8 | **Case study 1: LLM text completion** | decoding strategies, measured |
| 9 | **Case study 2: digital image processing** | five concrete uses |
| 10 | Shared mathematics, exercises, references | consolidation |

Everything is self-contained: no downloads, no API keys, and no GPU. Runtime is well under a minute on a laptop.

---
## 1. Setup

Requirements: `numpy`, `scipy`, `matplotlib`, `opencv-python`, `scikit-image`.

```bash
pip install numpy scipy matplotlib opencv-python scikit-image
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "figure.max_open_warning": 0})
np.set_printoptions(precision=4, suppress=True, linewidth=110)

RNG = np.random.default_rng(0)          # single seed source -> every result below is reproducible

def is_distribution(p, tol=1e-9):
    # A valid PMF: non-negative and sums to exactly 1 (within floating point tolerance)
    p = np.asarray(p, float)
    return bool((p >= -tol).all() and abs(p.sum() - 1.0) < tol)

def normalize(x):
    # Turn any non-negative score vector into a probability distribution
    x = np.asarray(x, float)
    return x / x.sum()

print("numpy", np.__version__, "| scipy", stats.__name__, "ready")

---
## 2. Random Variables, PMF, PDF, CDF

A **random variable** $X$ maps outcomes to numbers. Two cases matter here.

**Discrete** $X$ takes values in a countable set. Its **probability mass function** gives the probability of
each value:

$$p(x) = P(X = x), \qquad p(x) \ge 0, \qquad \sum_x p(x) = 1$$

**Continuous** $X$ takes values in $\mathbb{R}$. Its **probability density function** satisfies

$$f(x) \ge 0, \qquad \int_{-\infty}^{\infty} f(x)\,dx = 1, \qquad P(a \le X \le b) = \int_a^b f(x)\,dx$$

A density is **not** a probability: $f(x)$ can exceed 1. Only its integral over an interval is a probability.

The **cumulative distribution function** works for both and is the workhorse of this notebook:

$$F(x) = P(X \le x) = \sum_{t \le x} p(t) \quad \text{or} \quad \int_{-\infty}^{x} f(t)\,dt$$

$F$ is non-decreasing, right-continuous, and runs from 0 to 1. Section 5 shows that inverting $F$ is exactly
how tokens are sampled from a language model **and** how histogram equalization works.

**Expectation and variance:**

$$\mathbb{E}[X] = \sum_x x\,p(x), \qquad
\operatorname{Var}(X) = \mathbb{E}[(X-\mathbb{E}[X])^2] = \mathbb{E}[X^2] - \mathbb{E}[X]^2$$

In [ ]:
# A loaded six-sided die: an explicit categorical PMF
faces = np.arange(1, 7)
pmf = normalize([1, 1, 1, 1, 2, 4])          # face 6 is four times as likely as face 1
cdf = np.cumsum(pmf)

print("PMF :", pmf, " valid:", is_distribution(pmf))
print("CDF :", cdf)

mean = (faces * pmf).sum()
var = (faces**2 * pmf).sum() - mean**2
print(f"\nE[X]   = {mean:.4f}")
print(f"Var(X) = {var:.4f}   SD = {np.sqrt(var):.4f}")

# Empirical check: the law of large numbers in one line
draws = RNG.choice(faces, size=200_000, p=pmf)
print(f"\nempirical mean over 200k draws = {draws.mean():.4f}  (theory {mean:.4f})")
print(f"empirical var                  = {draws.var():.4f}  (theory {var:.4f})")

fig, ax = plt.subplots(1, 3, figsize=(12, 3))
ax[0].stem(faces, pmf); ax[0].set_title("PMF p(x)"); ax[0].set_xlabel("face")
ax[1].step(faces, cdf, where="post"); ax[1].set_ylim(0, 1.05)
ax[1].set_title("CDF F(x) = P(X <= x)"); ax[1].set_xlabel("face")
ax[2].hist(draws, bins=np.arange(0.5, 7.5), density=True, edgecolor="w")
ax[2].plot(faces, pmf, "ro", label="theoretical PMF"); ax[2].legend(fontsize=8)
ax[2].set_title("200k samples vs theory")
plt.tight_layout(); plt.show()

> **Reading the plots.** The empirical histogram converges to the PMF, but slowly: the error of a Monte Carlo
> estimate shrinks as $O(1/\sqrt{n})$. Going from 2,000 to 200,000 samples improves the estimate by only about
> a factor of 10. This is worth internalizing before trusting any sampled evaluation — including LLM benchmark
> scores computed from a few hundred generations.

---
## 3. Discrete Distribution Families

Each family below is a *mechanism*, not just a formula. Matching the mechanism to your problem is the actual
skill; the formula follows from it.

| Distribution | Mechanism it models | Parameters | Mean | Variance |
|---|---|---|---|---|
| Bernoulli$(p)$ | one yes/no trial | $p$ | $p$ | $p(1-p)$ |
| Binomial$(n,p)$ | count of successes in $n$ independent trials | $n, p$ | $np$ | $np(1-p)$ |
| **Categorical$(\boldsymbol{\pi})$** | one draw from $K$ labelled outcomes | $\pi_1..\pi_K$ | — | — |
| Poisson$(\lambda)$ | count of rare events in a fixed interval | $\lambda$ | $\lambda$ | $\lambda$ |
| Geometric$(p)$ | trials until the first success | $p$ | $1/p$ | $(1-p)/p^2$ |

Where they appear in the two case studies:

- **Categorical** — the next-token distribution of an LLM (Section 8) and the grayscale histogram of an image
  (Section 9). Same object, different $K$.
- **Bernoulli** — whether a given pixel is hit by impulse noise; whether a sampled token is accepted by a filter.
- **Poisson** — photon arrivals at a sensor pixel. This is why image noise grows with brightness (Section 9.3).
- **Binomial** — number of corrupted pixels in a patch; number of correct answers on a test.
- **Geometric** — number of rejection-sampling attempts before acceptance.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3))

k = np.arange(0, 21)
ax[0].stem(k, stats.binom.pmf(k, n=20, p=0.3))
ax[0].set_title("Binomial(n=20, p=0.3)")

for lam, m in [(1, "o"), (4, "s"), (10, "^")]:
    ax[1].plot(k, stats.poisson.pmf(k, lam), m + "-", ms=4, label=f"lambda={lam}")
ax[1].legend(fontsize=8); ax[1].set_title("Poisson: variance = mean")

kg = np.arange(1, 16)
for p, m in [(0.2, "o"), (0.5, "s")]:
    ax[2].plot(kg, stats.geom.pmf(kg, p), m + "-", ms=4, label=f"p={p}")
ax[2].legend(fontsize=8); ax[2].set_title("Geometric: trials until success")

cat = normalize(RNG.random(12) ** 3)          # a skewed categorical, like a token distribution
ax[3].bar(np.arange(12), np.sort(cat)[::-1])
ax[3].set_title("Categorical over K=12 (sorted)")
for a in ax:
    a.set_xlabel("k")
plt.tight_layout(); plt.show()

# The Poisson identity that drives sensor noise
for lam in [1, 4, 25, 100]:
    s = RNG.poisson(lam, 200_000)
    print(f"lambda={lam:4d}   sample mean={s.mean():7.3f}   sample var={s.var():7.3f}   "
          f"SD/mean={s.std()/s.mean():.3f}")
print("\nVariance tracks the mean, so absolute noise grows with brightness while RELATIVE noise (SD/mean) falls.")
print("That single fact explains why dark image regions look noisier than bright ones.")

---
## 4. Continuous Distribution Families

| Distribution | Density | Mechanism | Where it shows up |
|---|---|---|---|
| Uniform$(a,b)$ | $\frac{1}{b-a}$ | complete ignorance on an interval | quantization error, random init |
| Gaussian$(\mu,\sigma^2)$ | $\frac{1}{\sigma\sqrt{2\pi}}e^{-(x-\mu)^2/2\sigma^2}$ | sum of many small independent effects (CLT) | sensor read noise, logit distributions |
| Laplace$(\mu,b)$ | $\frac{1}{2b}e^{-|x-\mu|/b}$ | heavier tails than Gaussian | image gradients, sparse priors (L1) |
| Exponential$(\lambda)$ | $\lambda e^{-\lambda x}$ | waiting time between Poisson events | inter-arrival times |
| Rayleigh$(\sigma)$ | $\frac{x}{\sigma^2}e^{-x^2/2\sigma^2}$ | magnitude of a 2-D Gaussian vector | gradient magnitude, MRI/ultrasound noise |
| Beta$(\alpha,\beta)$ | $\propto x^{\alpha-1}(1-x)^{\beta-1}$ | a distribution **over probabilities** | Bayesian priors, smoothing |

Two of these deserve emphasis for the case studies.

**Gaussian, and why it is everywhere.** The Central Limit Theorem says that the normalized sum of many
independent finite-variance contributions converges to a Gaussian regardless of the individual shapes. Sensor
read noise is the sum of many small electrical effects, so it is Gaussian — not by assumption but by mechanism.

**Rayleigh, and why gradients are not Gaussian.** If $g_x, g_y$ are independent zero-mean Gaussians, then the
gradient magnitude $\sqrt{g_x^2+g_y^2}$ is Rayleigh-distributed. This is why the magnitude image from a Sobel
filter applied to pure noise has a characteristic skewed shape with a hard floor at zero, and it gives a
principled way to set an edge threshold: pick the quantile of the Rayleigh fit to the flat regions.

In [ ]:
x = np.linspace(-6, 6, 600)
xp = np.linspace(0, 6, 600)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].plot(x, stats.norm.pdf(x, 0, 1), label="Gaussian(0,1)")
ax[0].plot(x, stats.laplace.pdf(x, 0, 1 / np.sqrt(2)), label="Laplace (same variance)")
ax[0].legend(fontsize=8); ax[0].set_title("Same variance, very different tails")

ax[1].semilogy(x, stats.norm.pdf(x, 0, 1), label="Gaussian")
ax[1].semilogy(x, stats.laplace.pdf(x, 0, 1 / np.sqrt(2)), label="Laplace")
ax[1].legend(fontsize=8); ax[1].set_title("Log scale: tail behaviour")

ax[2].plot(xp, stats.expon.pdf(xp, scale=1), label="Exponential(1)")
ax[2].plot(xp, stats.rayleigh.pdf(xp, scale=1), label="Rayleigh(1)")
ax[2].legend(fontsize=8); ax[2].set_title("Positive-support densities")
plt.tight_layout(); plt.show()

# Tail mass: how often does each model produce a 4-sigma event?
print(f"P(|X| > 4 sigma), Gaussian : {2*stats.norm.sf(4):.3e}")
print(f"P(|X| > 4 sigma), Laplace  : {2*stats.laplace.sf(4/np.sqrt(2), scale=1/np.sqrt(2)):.3e}")
print("\nModelling heavy-tailed data with a Gaussian makes rare events look impossible.")
print("That is the failure mode behind outlier-blind estimators.")

In [ ]:
# CLT in action, and the Rayleigh fact behind gradient magnitudes
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))

for n, a in zip([1, 5, 30], ax):
    # Sum of n Uniform(0,1) variables, standardized -- a maximally non-Gaussian starting shape
    s = RNG.random((80_000, n)).sum(axis=1)
    s = (s - s.mean()) / s.std()
    a.hist(s, bins=80, density=True, alpha=0.75, edgecolor="none")
    a.plot(x, stats.norm.pdf(x), "r-", lw=1.5)
    a.set_title(f"mean of n={n} uniforms (standardized)")
    a.set_xlim(-4, 4)
plt.tight_layout(); plt.show()

gx = RNG.normal(0, 1, 200_000)
gy = RNG.normal(0, 1, 200_000)
mag = np.hypot(gx, gy)
plt.figure(figsize=(5.5, 3))
plt.hist(mag, bins=100, density=True, alpha=0.75, edgecolor="none", label="|gradient| of Gaussian noise")
plt.plot(xp, stats.rayleigh.pdf(xp, scale=1), "r-", lw=1.5, label="Rayleigh(sigma=1)")
plt.legend(fontsize=8); plt.title("Sobel magnitude on pure noise is Rayleigh, not Gaussian")
plt.tight_layout(); plt.show()
print(f"Rayleigh 99th percentile for sigma=1: {stats.rayleigh.ppf(0.99):.3f}")
print("-> an edge threshold at that value keeps the false-edge rate at about 1% in flat regions.")

---
## 5. Sampling and the Inverse-CDF Transform

This section is the bridge between the two case studies, so it is worth slowing down.

**Probability integral transform.** If $X$ has a continuous, strictly increasing CDF $F$, then

$$U = F(X) \sim \text{Uniform}(0,1)$$

and conversely, if $U \sim \text{Uniform}(0,1)$ then $X = F^{-1}(U)$ has CDF $F$.

Two consequences that look unrelated but are the same theorem:

1. **Sampling.** To draw from any distribution, draw $u \sim \text{Uniform}(0,1)$ and return $F^{-1}(u)$. For a
   categorical distribution this is "walk the cumulative sum until you pass $u$" — literally what an LLM
   decoder does at every step (Section 8).
2. **Histogram equalization.** Applying an image's own CDF as an intensity mapping, $s = F(r)$, makes the output
   intensities approximately uniform, which is the definition of histogram equalization (Section 9.2).

So token sampling and histogram equalization are the transform run in opposite directions: one uses $F^{-1}$ to
turn uniform noise into structure, the other uses $F$ to turn structure into a uniform spread.

In [ ]:
def sample_categorical(p, n, rng=RNG):
    # Inverse-CDF sampling, written out instead of calling rng.choice
    cdf = np.cumsum(p)
    u = rng.random(n)
    return np.searchsorted(cdf, u)          # index of the first cdf value >= u

p_test = normalize([0.05, 0.35, 0.10, 0.40, 0.10])
idx = sample_categorical(p_test, 100_000)
emp = np.bincount(idx, minlength=len(p_test)) / len(idx)
print("target      :", p_test)
print("empirical   :", emp)
print("max abs err :", np.abs(emp - p_test).max())

# The forward direction: F(X) is uniform, whatever X was
samples = RNG.exponential(scale=2.0, size=100_000)
u = stats.expon.cdf(samples, scale=2.0)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(samples, bins=80, density=True, edgecolor="none")
ax[0].set_title("X ~ Exponential(scale=2)")
ax[1].hist(u, bins=50, density=True, edgecolor="none")
ax[1].axhline(1.0, color="r", lw=1.5)
ax[1].set_title("U = F(X) is Uniform(0,1)")
plt.tight_layout(); plt.show()

---
## 6. Entropy, Cross-Entropy, KL Divergence, Perplexity

These four quantities are how distributions get **measured**. They are used identically in both case studies.

**Entropy** — the average surprise of a distribution, in bits (base 2) or nats (base $e$):

$$H(p) = -\sum_x p(x)\log p(x)$$

$H$ is maximal for the uniform distribution ($\log K$) and zero for a point mass. For an image, $H$ of the
intensity histogram is a lower bound on bits per pixel for any symbol-wise lossless code. For an LLM, $H$ of the
next-token distribution measures how undecided the model is at that step.

**Cross-entropy** — the cost of coding data from $p$ using a code designed for $q$:

$$H(p,q) = -\sum_x p(x)\log q(x)$$

This is exactly the training loss of a language model, with $p$ the one-hot true token and $q$ the model's
prediction.

**KL divergence** — the excess cost of that mismatch:

$$D_{KL}(p\,\|\,q) = \sum_x p(x)\log\frac{p(x)}{q(x)} = H(p,q) - H(p) \ge 0$$

It is zero only when $p = q$, and it is **not symmetric**: $D_{KL}(p\|q) \ne D_{KL}(q\|p)$ in general. The
asymmetry has practical meaning. $D_{KL}(p\|q)$ blows up when $q$ assigns near-zero probability to something
$p$ considers likely — which is why a language model must never assign exactly zero to any token, and why
n-gram models need smoothing (Section 8.2).

**Perplexity** — cross-entropy exponentiated back into "effective number of choices":

$$\mathrm{PPL} = \exp\!\big(H(p,q)\big) \quad \text{(nats)} \qquad\text{or}\qquad 2^{H(p,q)} \quad \text{(bits)}$$

A perplexity of 40 means the model is, on average, as uncertain as if it were choosing uniformly among 40 tokens.

In [ ]:
EPS = 1e-12

def entropy(p, base=2):
    p = np.asarray(p, float)
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / np.log(base))

def cross_entropy(p, q, base=2):
    p, q = np.asarray(p, float), np.asarray(q, float)
    return float(-(p * np.log(q + EPS)).sum() / np.log(base))

def kl_divergence(p, q, base=2):
    p, q = np.asarray(p, float), np.asarray(q, float)
    m = p > 0
    return float((p[m] * np.log(p[m] / (q[m] + EPS))).sum() / np.log(base))

def perplexity(p, q):
    return float(2 ** cross_entropy(p, q, base=2))

K = 8
uniform = np.full(K, 1 / K)
peaked = normalize([0.9, 0.04, 0.02, 0.01, 0.01, 0.01, 0.005, 0.005])
mild = normalize([0.30, 0.25, 0.15, 0.10, 0.08, 0.06, 0.04, 0.02])

print(f"{'distribution':<14}{'H (bits)':>10}{'max H':>8}{'perplexity 2^H':>17}")
print("-" * 49)
for name, d in [("uniform", uniform), ("mild", mild), ("peaked", peaked)]:
    print(f"{name:<14}{entropy(d):>10.4f}{np.log2(K):>8.2f}{2**entropy(d):>17.3f}")

print(f"\nKL(peaked || uniform) = {kl_divergence(peaked, uniform):.4f} bits")
print(f"KL(uniform || peaked) = {kl_divergence(uniform, peaked):.4f} bits   <- not symmetric")
print(f"KL(peaked || peaked)  = {kl_divergence(peaked, peaked):.4f} bits")
print(f"\nIdentity check  H(p,q) = H(p) + KL(p||q):")
print(f"  H(mild, peaked)          = {cross_entropy(mild, peaked):.6f}")
print(f"  H(mild) + KL(mild||peaked) = {entropy(mild) + kl_divergence(mild, peaked):.6f}")

In [ ]:
# Why zero probabilities are fatal: one unseen event makes cross-entropy diverge
q_zero = np.array([0.5, 0.5, 0.0, 0.0])
p_true = np.array([0.4, 0.4, 0.1, 0.1])
with np.errstate(divide="ignore"):
    naive = -(p_true * np.log2(np.where(q_zero > 0, q_zero, 0))).sum()
print("cross-entropy with a true zero in q :", naive)
print("perplexity                          :", 2.0 ** naive)
print("\nA single event the model called impossible destroys the average score.")
print("Fix: smoothing. Reserve a little mass for everything (Section 8.2).")

alpha = np.array([0.0, 0.01, 0.1, 0.5])
print(f"\n{'add-alpha':>10}{'q':>34}{'cross-entropy (bits)':>23}")
for a in alpha:
    q = normalize(np.array([0.5, 0.5, 0.0, 0.0]) * 4 + a)
    ce = cross_entropy(p_true, q) if (q > 0).all() else float("inf")
    print(f"{a:>10.2f}{str(np.round(q, 4)):>34}{ce:>23.4f}")

---
## 7. Estimation: Fitting Distributions to Data

Given data, which parameters make it most plausible?

**Maximum likelihood (MLE).** Choose $\theta$ maximizing $\prod_i p(x_i \mid \theta)$, equivalently the
log-likelihood $\sum_i \log p(x_i \mid \theta)$. For a Gaussian this gives the familiar closed form
$\hat\mu = \bar{x}$, $\hat\sigma^2 = \frac{1}{n}\sum (x_i-\bar{x})^2$. For a categorical it gives the
**empirical frequency** — counts divided by total. That is why an image histogram *is* the MLE of its intensity
distribution, and why an n-gram model's raw counts are its MLE.

**Maximum a posteriori (MAP).** Add a prior and maximize $p(\theta \mid x) \propto p(x\mid\theta)\,p(\theta)$.
For categorical data with a Dirichlet/Beta prior, MAP is exactly **add-$\alpha$ smoothing**: pretend you saw
$\alpha$ extra occurrences of every symbol before looking at the data. The pseudo-count is not a hack; it is a
prior.

$$\hat\pi_k^{\text{MAP}} = \frac{c_k + \alpha}{\sum_j (c_j + \alpha)}$$

MLE is MAP with a flat prior and infinite confidence in the data. With scarce data — rare tokens, rare
intensities — that confidence is misplaced, which is exactly what Section 8.2 measures.

In [ ]:
# MLE for a Gaussian, and how the estimate tightens with n
true_mu, true_sigma = 128.0, 12.0
print(f"{'n':>8}{'mu_hat':>10}{'sigma_hat':>12}{'|error| mu':>13}")
print("-" * 43)
for n in [10, 100, 1_000, 100_000]:
    s = RNG.normal(true_mu, true_sigma, n)
    mu_hat, sd_hat = s.mean(), s.std(ddof=0)
    print(f"{n:>8}{mu_hat:>10.3f}{sd_hat:>12.3f}{abs(mu_hat-true_mu):>13.4f}")

# MAP = add-alpha smoothing for a categorical, shown on a tiny count vector
counts = np.array([7, 2, 1, 0, 0])
print(f"\ncounts observed: {counts}   (two symbols never seen)")
print(f"{'alpha':>7}{'estimated distribution':>42}{'P(unseen)':>12}")
for a in [0.0, 0.1, 0.5, 1.0]:
    est = (counts + a) / (counts + a).sum()
    print(f"{a:>7.1f}{str(np.round(est, 4)):>42}{est[-1]:>12.4f}")
print("\nalpha = 0 (pure MLE) declares the unseen symbols impossible -- a claim 10 observations cannot support.")

---
# 8. Case Study 1 — Text Completion in a Language Model

## 8.1 What an LLM actually outputs

A transformer does **not** output text. At each position it outputs a vector of real numbers, the **logits**
$z \in \mathbb{R}^{V}$, one per vocabulary entry. Those are converted into a categorical distribution by the
softmax:

$$q_i = \frac{e^{z_i / T}}{\sum_{j=1}^{V} e^{z_j / T}}$$

Three properties matter and all three are visible in the code below:

1. **Shift invariance.** $\mathrm{softmax}(z + c) = \mathrm{softmax}(z)$. Only *differences* between logits
   carry information. Implementations subtract $\max_j z_j$ before exponentiating to avoid overflow.
2. **Temperature $T$** rescales the logits. $T \to 0$ concentrates all mass on the argmax (greedy decoding);
   $T = 1$ is the model's own distribution; $T \to \infty$ approaches uniform. Temperature does not add
   knowledge — it only redistributes mass the model already assigned.
3. **Full support.** Because $e^{z}>0$ always, softmax gives *every* token non-zero probability. Truncation
   methods (top-k, top-p) exist to remove the long tail of tokens the model does not actually endorse.

Everything after the transformer — greedy, temperature sampling, top-k, top-p, beam search — is a decision rule
applied to this categorical distribution. That is why this case study can be run faithfully with a small n-gram
model instead of a transformer: the mathematics of decoding is identical; only the quality of $q$ differs.

In [ ]:
def softmax(z, T=1.0):
    z = np.asarray(z, float) / T
    z = z - z.max()                     # shift for numerical stability; does not change the result
    e = np.exp(z)
    return e / e.sum()

logits = np.array([3.2, 1.8, 1.5, 0.4, -0.7, -2.1])
tokens = ["Kendari", "Jakarta", "Makassar", "the", "quantum", "zzz"]

print("shift invariance:", np.allclose(softmax(logits), softmax(logits + 1000)))
print()
print(f"{'T':>6}" + "".join(f"{t:>11}" for t in tokens) + f"{'entropy':>10}")
print("-" * 82)
for T in [0.01, 0.3, 0.7, 1.0, 1.5, 3.0, 100.0]:
    q = softmax(logits, T)
    print(f"{T:>6.2f}" + "".join(f"{v:>11.4f}" for v in q) + f"{entropy(q):>10.3f}")
print(f"\nuniform entropy for V={len(tokens)} is log2(V) = {np.log2(len(tokens)):.3f} bits")

Ts = np.linspace(0.05, 5, 120)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(Ts, [entropy(softmax(logits, t)) for t in Ts])
ax[0].axhline(np.log2(len(tokens)), color="r", ls="--", label="uniform bound")
ax[0].set_xlabel("temperature T"); ax[0].set_ylabel("entropy (bits)")
ax[0].legend(fontsize=8); ax[0].set_title("Temperature controls uncertainty")
for i, t in enumerate(tokens):
    ax[1].plot(Ts, [softmax(logits, x)[i] for x in Ts], label=t)
ax[1].set_xlabel("temperature T"); ax[1].set_ylabel("probability")
ax[1].legend(fontsize=7); ax[1].set_title("Mass flows from the top token to the tail")
plt.tight_layout(); plt.show()

## 8.2 A real (small) language model

To keep everything reproducible and offline, we train a **character-level n-gram model** on a short corpus
embedded in the notebook. An n-gram model estimates

$$q(x_t \mid x_{t-n+1:t-1}) = \frac{c(x_{t-n+1:t}) + \alpha}{\sum_{v} \big(c(x_{t-n+1:t-1}, v) + \alpha\big)}$$

which is the MAP estimate from Section 7: counts plus a Dirichlet pseudo-count $\alpha$. It is a genuine
language model — it defines a probability distribution over the next symbol given context, is trained by
maximum likelihood, and is evaluated by perplexity — it simply has a tiny context window and no ability to
generalize. Every decoding experiment below transfers unchanged to a transformer.

The implementation also supports **backoff**: if the full $n-1$ character context was never seen in training,
fall back to the $n-2$ context, and so on down to the unigram distribution. Backoff is switched off for the
smoothing experiment in 8.3 (so the failure mode of $\alpha = 0$ stays visible) and on for generation in 8.4,
where an unseen context otherwise collapses the model to near-uniform noise.

In [ ]:
CORPUS = """
the university admissions system verifies each applicant document before the selection committee meets.
the committee reviews the transcript, the diploma, and the identity card of every applicant.
each document is scanned, enhanced, and stored in the archive of the university.
the system computes a score for every applicant and ranks the applicants by that score.
an applicant with an incomplete document receives a notification and may upload the document again.
the archive stores every scanned document together with the checksum of the file.
the selection committee publishes the result after the verification of every document is complete.
a scanned diploma with low contrast is enhanced before the text of the diploma is extracted.
the identity card of an applicant is verified against the national registry of the ministry.
every applicant receives a receipt after the registration of the application is complete.
"""

# Train/test split by lines, so evaluation is on genuinely unseen text
lines = [l.strip() for l in CORPUS.strip().split("\n") if l.strip()]
train_text = " ".join(lines[:8])
test_text = " ".join(lines[8:])
VOCAB = sorted(set(train_text + test_text))
STOI = {c: i for i, c in enumerate(VOCAB)}
V = len(VOCAB)
print(f"vocabulary V = {V} characters: {''.join(VOCAB)!r}")
print(f"train: {len(train_text)} chars | test: {len(test_text)} chars")


class NGramLM:
    # Character-level n-gram language model, add-alpha (Dirichlet) smoothing, optional backoff
    def __init__(self, n=4, alpha=0.1, backoff=False):
        self.n, self.alpha, self.backoff = n, alpha, backoff
        self.counts = {}                       # counts[order][context] -> count vector over V

    def fit(self, text):
        t = " " * (self.n - 1) + text
        for order in range(1, self.n + 1):     # store every order so backoff has somewhere to fall
            table = self.counts.setdefault(order, {})
            for i in range(order - 1, len(t)):
                ctx = t[i - order + 1:i]
                table.setdefault(ctx, np.zeros(V))[STOI[t[i]]] += 1
        return self

    def predict(self, context):
        # Full categorical distribution over the vocabulary for the next character
        orders = range(self.n, 0, -1) if self.backoff else [self.n]
        for o in orders:
            ctx = (" " * (o - 1) + context)[-(o - 1):] if o > 1 else ""
            c = self.counts[o].get(ctx)
            if c is not None and c.sum() > 0:
                q = c + self.alpha
                return q / q.sum()
        # Context unseen at every usable order: fall back to the smoothing prior alone.
        # With alpha = 0 there is no prior, so the model genuinely assigns zero everywhere.
        return np.full(V, 1 / V) if self.alpha > 0 else np.zeros(V)

    def logits(self, context):
        # Log-probabilities play the role of transformer logits (softmax(log q) == q)
        return np.log(self.predict(context) + 1e-300)


lm = NGramLM(n=5, alpha=0.05, backoff=True).fit(train_text)
print("context tables (order, number of distinct contexts):",
      [(o, len(t)) for o, t in sorted(lm.counts.items())])

ctx = "the do"
q = lm.predict(ctx)
top = np.argsort(q)[::-1][:8]
print(f"\nnext-character distribution after {ctx!r}:")
for i in top:
    print(f"   {VOCAB[i]!r:>5}  p = {q[i]:.4f}  {'#' * int(60 * q[i])}")
print(f"\nentropy at this step: {entropy(q):.3f} bits  (uniform would be {np.log2(V):.3f})")
print("softmax(logits) reproduces the distribution:", np.allclose(softmax(lm.logits(ctx)), q))

## 8.3 Perplexity, and why smoothing is not optional

Perplexity on held-out text is the standard intrinsic evaluation of a language model:

$$\mathrm{PPL} = \exp\left(-\frac{1}{N}\sum_{t=1}^{N} \ln q(x_t \mid x_{<t})\right)$$

Because the true next character is a one-hot distribution, cross-entropy reduces to the negative log-probability
the model assigned to the character that actually occurred. The table below sweeps $\alpha$ and $n$: it shows the
bias–variance trade-off directly, and it shows what happens at $\alpha = 0$ (pure MLE) when the test set contains
a context–character pair the training set never produced.

In [ ]:
def perplexity_on(model, text):
    # Average negative log-probability of the actual next character, exponentiated.
    # A single zero-probability event makes the whole score infinite -- that is not a bug.
    nll = 0.0
    for i, ch in enumerate(text):
        p = model.predict(text[:i])[STOI[ch]]
        if p <= 0:
            return float("inf")
        nll -= np.log(p)
    return float(np.exp(nll / len(text)))


print("Fixed-order model (no backoff):")
print(f"{'n':>3}{'alpha':>8}{'train PPL':>12}{'test PPL':>12}   comment")
print("-" * 62)
for n in [2, 3, 4, 5]:
    for a in [0.0, 0.01, 0.05, 0.2]:
        m = NGramLM(n=n, alpha=a).fit(train_text)
        tr, te = perplexity_on(m, train_text), perplexity_on(m, test_text)
        note = "zero probability given to an observed event" if np.isinf(te) else ""
        print(f"{n:>3}{a:>8.2f}{tr:>12.2f}{te:>12.2f}   {note}")

print("\nSame models, with backoff to shorter contexts:")
print(f"{'n':>3}{'alpha':>8}{'train PPL':>12}{'test PPL':>12}")
print("-" * 35)
for n in [2, 3, 4, 5]:
    for a in [0.01, 0.05, 0.2]:
        m = NGramLM(n=n, alpha=a, backoff=True).fit(train_text)
        print(f"{n:>3}{a:>8.2f}{perplexity_on(m, train_text):>12.2f}"
              f"{perplexity_on(m, test_text):>12.2f}")

print("\nThree things to read off these tables:")
print("1. alpha = 0 is unusable: one unseen context-character pair sends test perplexity to infinity.")
print("2. Train perplexity keeps improving with larger n while test perplexity turns around: memorization.")
print("3. Backoff and smoothing address overlapping problems; both trade train fit for test robustness.")

## 8.4 Decoding strategies

The model gives $q$. A decoding strategy turns $q$ into an actual token. Four standard rules:

| Strategy | Rule | Failure mode |
|---|---|---|
| **Greedy** | $\arg\max_i q_i$ | deterministic loops; bland text |
| **Temperature** | sample from $\mathrm{softmax}(z/T)$ | high $T$ produces incoherence; low $T$ approaches greedy |
| **Top-k** | keep the $k$ highest, renormalize, sample | fixed $k$ ignores how peaked $q$ is at this step |
| **Top-p (nucleus)** | keep the smallest set with cumulative mass $\ge p$, renormalize, sample | adaptive; $p$ still needs tuning |

The key argument for **top-p over top-k**: the appropriate number of plausible continuations varies enormously
between steps. After "the do" only a couple of characters are plausible; after a space, dozens are. Top-k uses
the same width everywhere; top-p adapts its width to the entropy of the step. Section 8.5 measures exactly that.

In [ ]:
def apply_top_k(q, k):
    if k is None or k >= len(q):
        return q.copy()
    out = np.zeros_like(q)
    idx = np.argpartition(q, -k)[-k:]
    out[idx] = q[idx]
    return out / out.sum()


def apply_top_p(q, p):
    if p is None or p >= 1.0:
        return q.copy()
    order = np.argsort(q)[::-1]
    cum = np.cumsum(q[order])
    keep = order[:np.searchsorted(cum, p) + 1]     # smallest set whose mass reaches p
    out = np.zeros_like(q)
    out[keep] = q[keep]
    return out / out.sum()


def generate(model, prompt, n_chars=110, T=1.0, top_k=None, top_p=None, greedy=False, rng=None):
    rng = rng or np.random.default_rng(7)
    text = prompt
    for _ in range(n_chars):
        q = softmax(model.logits(text), T)
        q = apply_top_k(q, top_k)
        q = apply_top_p(q, top_p)
        j = int(np.argmax(q)) if greedy else int(sample_categorical(q, 1, rng)[0])
        text += VOCAB[j]
    return text


PROMPT = "the university "
settings = [("greedy (argmax)", dict(greedy=True)),
            ("T = 0.5", dict(T=0.5)),
            ("T = 1.0 (raw model)", dict(T=1.0)),
            ("T = 1.5", dict(T=1.5)),
            ("T = 1.0 + top-k=3", dict(T=1.0, top_k=3)),
            ("T = 1.0 + top-p=0.9", dict(T=1.0, top_p=0.9)),
            ("T = 1.5 + top-p=0.9", dict(T=1.5, top_p=0.9))]

for name, kw in settings:
    out = generate(lm, PROMPT, rng=np.random.default_rng(7), **kw)
    print(f"[{name}]\n  {out}\n")

> **Read the outputs, not the labels.** Four things are visible:
>
> - **Greedy** starts coherently and then enters a repeating cycle. A deterministic rule applied to a fixed
>   distribution has nowhere else to go; this is the same degeneracy that makes production LLMs loop at
>   temperature 0, and it is the motivating example in the nucleus sampling paper.
> - **Low temperature** stays close to real words because it suppresses the tail, at the cost of recycling
>   training phrases almost verbatim.
> - **Raising $T$** buys diversity and pays in spelling; by $T = 1.5$ the character stream is close to noise.
> - **Truncation helps, but it cannot rescue a weak model.** `T = 1.0 + top-k=3` is the most word-like output
>   here, yet `T = 1.5 + top-p=0.9` is still poor. Truncation restricts *which* symbols are reachable; it does
>   not improve the ranking the model produced. Better text needs a better $q$, not a better sampler — which is
>   the honest limit of every decoding-parameter debate.

In [ ]:
# How many characters does each strategy actually leave available, step by step?
def nucleus_size(q, p):
    cum = np.cumsum(np.sort(q)[::-1])
    return int(np.searchsorted(cum, p) + 1)

probe_text = "the applicant receives a notification about the scanned document"
rows = []
for i in range(4, len(probe_text)):
    q = lm.predict(probe_text[:i])
    rows.append((probe_text[i - 3:i], entropy(q), nucleus_size(q, 0.9), q.max()))

sizes = np.array([r[2] for r in rows])
ents = np.array([r[1] for r in rows])

print(f"nucleus size at p=0.9 over {len(rows)} decoding steps:")
print(f"  min={sizes.min()}  median={int(np.median(sizes))}  max={sizes.max()}  mean={sizes.mean():.2f}")
print(f"  correlation(entropy, nucleus size) = {np.corrcoef(ents, sizes)[0,1]:.3f}")
print("\nA fixed top-k must pick one number for all of these steps. Top-p picks it per step.\n")

print(f"{'context':>10}{'entropy':>10}{'|nucleus|':>11}{'max q':>9}")
print("-" * 40)
for ctx3, h, s, mx in rows[:6] + rows[-6:]:
    print(f"{ctx3!r:>10}{h:>10.3f}{s:>11}{mx:>9.3f}")

plt.figure(figsize=(6, 3))
plt.scatter(ents, sizes, s=14, alpha=0.6)
plt.xlabel("entropy of q at this step (bits)"); plt.ylabel("nucleus size at p=0.9")
plt.title("Top-p adapts its width to step-level uncertainty")
plt.tight_layout(); plt.show()

## 8.5 Measuring what truncation does to the distribution

Truncation is a *modification of the model's distribution*. KL divergence quantifies exactly how much
modification each setting applies, averaged over decoding steps. This gives an objective way to compare
settings that otherwise get compared by vibes.

In [ ]:
contexts = [probe_text[:i] for i in range(4, len(probe_text))]
base = [lm.predict(c) for c in contexts]

configs = [("T=0.3", lambda q: softmax(np.log(q), 0.3)),
           ("T=0.7", lambda q: softmax(np.log(q), 0.7)),
           ("T=1.0 (identity)", lambda q: q),
           ("T=1.5", lambda q: softmax(np.log(q), 1.5)),
           ("top-k=3", lambda q: apply_top_k(q, 3)),
           ("top-k=10", lambda q: apply_top_k(q, 10)),
           ("top-p=0.9", lambda q: apply_top_p(q, 0.9)),
           ("top-p=0.95", lambda q: apply_top_p(q, 0.95)),
           ("T=1.5 + top-p=0.9", lambda q: apply_top_p(softmax(np.log(q), 1.5), 0.9))]

print(f"{'setting':<20}{'mean H (bits)':>15}{'mean KL(q_mod||q)':>20}{'mean support':>14}")
print("-" * 69)
for name, f in configs:
    mods = [f(q) for q in base]
    mH = np.mean([entropy(m) for m in mods])
    mKL = abs(np.mean([kl_divergence(m, q) for m, q in zip(mods, base)]))
    sup = np.mean([(m > 1e-12).sum() for m in mods])
    print(f"{name:<20}{mH:>15.3f}{mKL:>20.4f}{sup:>14.1f}")

print("\nKL is computed as KL(modified || original): the extra cost of the symbols the modified")
print("distribution now prefers. Note that KL alone does NOT separate the two mechanisms --")
print("T=0.3 and top-k=3 land at similar divergence. The support column is what distinguishes them:")
print("temperature reaches its divergence by redistributing mass over all 26 symbols, truncation")
print("reaches a comparable divergence by deleting most of them. Same distance, different geometry,")
print("which is why production decoders expose both knobs instead of one.")

> **Summary of Case Study 1.** Every decoding knob is a transformation of one categorical distribution:
>
> - **temperature** reshapes it (entropy up or down, support unchanged),
> - **top-k / top-p** truncate it (support shrinks, tail mass redistributed),
> - **greedy** collapses it to a point mass,
> - **perplexity** scores the underlying distribution rather than any one sample from it.
>
> The practical consequence: quality complaints about an LLM ("too repetitive", "too random", "hallucinating
> rare words") map onto measurable properties of $q$ — entropy, nucleus size, tail mass — and can be diagnosed
> with the code above before anyone touches the model itself.

---
# 9. Case Study 2 — Probability Distributions in Digital Image Processing

A grayscale image is a sample from a distribution over $\{0,\dots,255\}$, and its **histogram is the maximum
likelihood estimate of that distribution** (Section 7). Once that is accepted, five standard image processing
operations stop being recipes and become consequences.

| Subsection | Operation | The distribution idea behind it |
|---|---|---|
| 9.2 | Histogram equalization | probability integral transform, $s = F(r)$ |
| 9.3 | Noise identification | matching a noise family to its mechanism |
| 9.4 | Otsu thresholding | two-component mixture; between-class variance |
| 9.5 | Bayesian pixel classification | posterior $\propto$ likelihood $\times$ prior |
| 9.6 | Compression bounds | entropy as bits per pixel |

We use two images: `skimage.data.camera` (a natural photograph) and a synthetic scanned-document patch, because
documents and photographs have very different histogram shapes and that difference drives every decision below.

In [ ]:
import cv2
from skimage import data

def make_document(h=320, w=520, seed=3, ink=70, ink_sd=22, paper=215, paper_sd=11, blur=0.7):
    # Synthetic scanned document: ink and paper are two Gaussian classes, blurred by the
    # scanner point spread function so intermediate intensities exist at every stroke edge.
    rng = np.random.default_rng(seed)
    img = np.full((h, w), float(paper))
    for r in range(6):                                   # text lines: dark bars with gaps
        y = 30 + r * 45
        x = 40
        while x < w - 60:
            wgap = rng.integers(8, 34)
            img[y:y + 16, x:x + wgap] = rng.normal(ink, ink_sd * 0.6)
            x += wgap + rng.integers(6, 14)
    img = cv2.GaussianBlur(img, (0, 0), blur)            # optical blur -> soft edges
    is_ink = img < (ink + paper) / 2
    img = img + np.where(is_ink, rng.normal(0, ink_sd * 0.8, img.shape),
                         rng.normal(0, paper_sd, img.shape))
    return np.clip(img, 0, 255).astype(np.uint8)


CAMERA = data.camera()
DOC = make_document()

def hist_pmf(img, bins=256):
    # The MLE of the intensity distribution = normalized histogram
    h = np.bincount(img.ravel(), minlength=bins).astype(float)
    return h / h.sum()

fig, ax = plt.subplots(2, 3, figsize=(13, 5.6))
for row, (img, name) in enumerate([(CAMERA, "camera (photograph)"), (DOC, "synthetic document")]):
    p = hist_pmf(img)
    ax[row, 0].imshow(img, cmap="gray", vmin=0, vmax=255); ax[row, 0].set_title(name)
    ax[row, 0].axis("off"); ax[row, 0].grid(False)
    ax[row, 1].bar(np.arange(256), p, width=1.0)
    ax[row, 1].set_title(f"PMF (histogram), H = {entropy(p):.3f} bits")
    ax[row, 2].plot(np.cumsum(p)); ax[row, 2].set_ylim(0, 1.02)
    ax[row, 2].set_title("CDF")
plt.tight_layout(); plt.show()

print(f"{'image':<22}{'entropy (bits/px)':>19}{'mean':>9}{'sd':>9}")
print("-" * 59)
for img, name in [(CAMERA, "camera"), (DOC, "document")]:
    p = hist_pmf(img)
    print(f"{name:<22}{entropy(p):>19.4f}{img.mean():>9.2f}{img.std():>9.2f}")
print("\nThe document histogram is bimodal -- two classes, ink and paper -- while the photograph")
print("spreads its mass across the whole range. Note that the document entropy is not dramatically")
print("lower here, because sensor noise widens each of the two modes. Section 9.6 shows the same")
print("layout without noise at about 2.3 bits/pixel: the structure is cheap, the noise is expensive.")

## 9.2 Histogram equalization is the probability integral transform

Section 5 proved that $F(X)$ is uniform when $F$ is the CDF of $X$. Applying that to intensities gives the
classical transformation

$$s = T(r) = (L-1)\,F(r) = (L-1)\sum_{i=0}^{r} p(i)$$

with $L = 256$. The output is *approximately* uniform — approximately, not exactly, because intensities are
discrete: all pixels sharing an input level must share an output level, so mass cannot be split. Equalization
can therefore never increase entropy; at best it preserves it while spreading the values apart.

This matters for documents. A bimodal document histogram has two tall spikes, and equalization drags them apart
without creating any new detail — it can even make the image look worse while the numbers say "more contrast".
The measurement below shows both effects.

In [ ]:
def equalize(img):
    p = hist_pmf(img)
    lut = np.round(255 * np.cumsum(p)).astype(np.uint8)     # T(r) = 255 * F(r)
    return lut[img], lut

fig, ax = plt.subplots(2, 4, figsize=(15, 5.6))
for row, (img, name) in enumerate([(CAMERA, "camera"), (DOC, "document")]):
    eq, lut = equalize(img)
    p_in, p_out = hist_pmf(img), hist_pmf(eq)
    ax[row, 0].imshow(img, cmap="gray", vmin=0, vmax=255); ax[row, 0].set_title(f"{name}: original")
    ax[row, 1].imshow(eq, cmap="gray", vmin=0, vmax=255); ax[row, 1].set_title("equalized")
    for a in ax[row, :2]:
        a.axis("off"); a.grid(False)
    ax[row, 2].plot(lut); ax[row, 2].set_title("transfer function s = 255 F(r)")
    ax[row, 2].set_xlabel("input r"); ax[row, 2].set_ylabel("output s")
    ax[row, 3].bar(np.arange(256), p_out, width=1.0)
    ax[row, 3].set_title(f"output PMF, H = {entropy(p_out):.3f} bits")
plt.tight_layout(); plt.show()

print(f"{'image':<12}{'H before':>10}{'H after':>10}{'levels before':>15}{'levels after':>14}")
print("-" * 61)
for img, name in [(CAMERA, "camera"), (DOC, "document")]:
    eq, _ = equalize(img)
    print(f"{name:<12}{entropy(hist_pmf(img)):>10.4f}{entropy(hist_pmf(eq)):>10.4f}"
          f"{len(np.unique(img)):>15}{len(np.unique(eq)):>14}")
print("\nEntropy never increases and the number of distinct levels never increases:")
print("equalization is a many-to-one mapping, so it redistributes information, it cannot create it.")

## 9.3 Identifying noise from its distribution

Enhancement choices depend on which noise family is present, and each family comes from a different physical
mechanism (Sections 3 and 4):

| Noise | Mechanism | Signature in the data |
|---|---|---|
| Gaussian | thermal / read noise, sum of many small effects | symmetric histogram in a flat patch; variance independent of brightness |
| Poisson (shot) | photon counting | **variance grows linearly with mean** |
| Impulse (salt & pepper) | dead pixels, transmission errors | isolated spikes at 0 and 255 |
| Speckle / Rayleigh | coherent imaging (ultrasound, SAR) | multiplicative, skewed, positive support |

The practical procedure: crop a **flat region**, look at its histogram, and measure the mean–variance
relationship across patches of different brightness. A flat mean–variance plot means additive Gaussian; a line
through the origin with slope $g$ means Poisson with sensor gain $g$. This is the *photon transfer curve*, and
it is how camera gain is measured in practice.

In [ ]:
flat = DOC[5:25, 300:500].astype(float)          # a paper-only patch: no structure, only noise

noisy = {
    "gaussian sigma=12": np.clip(DOC + RNG.normal(0, 12, DOC.shape), 0, 255).astype(np.uint8),
    "poisson (shot)": np.clip(RNG.poisson(DOC.astype(float) / 255 * 60) / 60 * 255, 0, 255).astype(np.uint8),
    "salt & pepper 5%": DOC.copy(),
    "speckle": np.clip(DOC * RNG.normal(1.0, 0.18, DOC.shape), 0, 255).astype(np.uint8),
}
m = RNG.random(DOC.shape)
noisy["salt & pepper 5%"][m < 0.025] = 0
noisy["salt & pepper 5%"][m > 0.975] = 255

fig, ax = plt.subplots(2, 4, figsize=(15, 5.2))
for j, (name, img) in enumerate(noisy.items()):
    ax[0, j].imshow(img, cmap="gray", vmin=0, vmax=255); ax[0, j].set_title(name)
    ax[0, j].axis("off"); ax[0, j].grid(False)
    patch = img[5:25, 300:500].ravel()
    ax[1, j].hist(patch, bins=60, density=True, edgecolor="none")
    ax[1, j].set_title(f"flat-patch histogram\nmean={patch.mean():.1f} sd={patch.std():.1f}")
plt.tight_layout(); plt.show()

# Photon transfer curve: variance vs mean over patches of different brightness
levels = np.array([15, 40, 80, 130, 180, 230], float)
gauss_pts, pois_pts = [], []
for L in levels:
    base = np.full((160, 160), L)
    g = base + RNG.normal(0, 12, base.shape)
    pz = RNG.poisson(base / 255 * 60) / 60 * 255
    gauss_pts.append((g.mean(), g.var()))
    pois_pts.append((pz.mean(), pz.var()))
gauss_pts, pois_pts = np.array(gauss_pts), np.array(pois_pts)
slope = np.polyfit(pois_pts[:, 0], pois_pts[:, 1], 1)[0]

plt.figure(figsize=(6, 3.2))
plt.plot(gauss_pts[:, 0], gauss_pts[:, 1], "o-", label="additive Gaussian: flat")
plt.plot(pois_pts[:, 0], pois_pts[:, 1], "s-", label=f"Poisson: slope = {slope:.2f}")
plt.xlabel("patch mean"); plt.ylabel("patch variance")
plt.title("Photon transfer curve identifies the noise mechanism"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Diagnosis rule: flat line -> additive Gaussian, denoise uniformly.")
print("Rising line   -> shot noise; either denoise adaptively or apply a variance-stabilizing")
print("transform (Anscombe, x -> 2*sqrt(x + 3/8)) first, which makes the variance constant.")

In [ ]:
# Maximum likelihood estimate of the noise level from a flat region, and why the estimator matters
patch = noisy["gaussian sigma=12"][5:25, 300:500].astype(float)
mle_sigma = patch.std(ddof=0)
robust_sigma = 1.4826 * np.median(np.abs(patch - np.median(patch)))   # MAD estimator

sp_patch = noisy["salt & pepper 5%"][5:25, 300:500].astype(float)
print(f"{'patch':<26}{'MLE sd':>10}{'robust (MAD) sd':>18}")
print("-" * 54)
print(f"{'gaussian sigma=12':<26}{mle_sigma:>10.2f}{robust_sigma:>18.2f}")
print(f"{'salt & pepper 5%':<26}{sp_patch.std(ddof=0):>10.2f}"
      f"{1.4826*np.median(np.abs(sp_patch-np.median(sp_patch))):>18.2f}")
print("\nThe measured scale is larger than the injected sigma=12 because the paper already carried")
print("its own texture (sd about 11); independent noise sources add in quadrature: sqrt(11^2+12^2) = 16.3.")
print("On this Gaussian-only patch both estimators agree. With 5% impulses, the MLE (which assumes")
print("a Gaussian) is destroyed by the outliers while the MAD estimator is barely affected --")
print("a concrete instance of the heavy-tail warning from Section 4.")

## 9.4 Otsu's threshold is a two-component mixture model

Binarizing a document assumes the intensity distribution is a **mixture of two classes**, ink and paper:

$$p(r) = \omega_0\, p_0(r) + \omega_1\, p_1(r)$$

Otsu's method searches every threshold $t$ and maximizes the **between-class variance**

$$\sigma_B^2(t) = \omega_0(t)\,\omega_1(t)\,\big[\mu_0(t) - \mu_1(t)\big]^2$$

which is equivalent to minimizing the within-class variance because the total variance is fixed. Implemented
directly below, it is a five-line algorithm — and it is exactly a hard-assignment fit of a two-component
Gaussian mixture with equal variances. Fitting a proper GMM with EM gives soft assignments and unequal
variances, which matters when the two classes have very different spreads.

Otsu's assumption also tells you when it will fail: **if the histogram is not bimodal, there is no valid
threshold to find.** That is the real reason global thresholding collapses under uneven illumination.

In [ ]:
def otsu_threshold(img):
    p = hist_pmf(img)
    levels = np.arange(256)
    omega0 = np.cumsum(p)                                # class 0 weight for each t
    mu = np.cumsum(p * levels)
    mu_T = mu[-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        sigma_b = (mu_T * omega0 - mu) ** 2 / (omega0 * (1 - omega0))
    sigma_b = np.nan_to_num(sigma_b)
    t = int(np.argmax(sigma_b))
    return t, sigma_b

t_doc, sb_doc = otsu_threshold(DOC)
t_cam, sb_cam = otsu_threshold(CAMERA)
t_cv, _ = cv2.threshold(DOC, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"document: our Otsu t = {t_doc}, OpenCV t = {int(t_cv)}   camera: t = {t_cam}")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
p = hist_pmf(DOC)
ax[0].bar(np.arange(256), p, width=1.0); ax[0].axvline(t_doc, color="r")
ax[0].set_title(f"document histogram, Otsu t={t_doc}")
ax[1].plot(sb_doc); ax[1].axvline(t_doc, color="r")
ax[1].set_title("between-class variance sigma_B^2(t)")
ax[2].imshow(DOC > t_doc, cmap="gray"); ax[2].set_title("binarized"); ax[2].axis("off"); ax[2].grid(False)
plt.tight_layout(); plt.show()

# EM for a 2-component Gaussian mixture on the same histogram
def fit_gmm_1d(img, iters=60, seed=0):
    x = img.ravel().astype(float)
    rng = np.random.default_rng(seed)
    mu = np.array([x.min() + 20.0, x.max() - 20.0])
    sd = np.array([25.0, 25.0])
    w = np.array([0.5, 0.5])
    for _ in range(iters):
        # E step: responsibilities
        r = np.stack([w[k] * stats.norm.pdf(x, mu[k], sd[k]) for k in range(2)])
        r /= r.sum(axis=0, keepdims=True) + 1e-300
        # M step: weighted MLE
        nk = r.sum(axis=1)
        w = nk / nk.sum()
        mu = (r * x).sum(axis=1) / nk
        sd = np.sqrt((r * (x - mu[:, None]) ** 2).sum(axis=1) / nk) + 1e-6
    return w, mu, sd

w, mu_g, sd_g = fit_gmm_1d(DOC)
print(f"\nGMM fit: weights={np.round(w,3)}  means={np.round(mu_g,2)}  sds={np.round(sd_g,2)}")

grid = np.arange(256)
mix = w[0] * stats.norm.pdf(grid, mu_g[0], sd_g[0]) + w[1] * stats.norm.pdf(grid, mu_g[1], sd_g[1])
plt.figure(figsize=(6.5, 3))
plt.bar(grid, hist_pmf(DOC), width=1.0, label="histogram (MLE)")
plt.plot(grid, mix, "r-", lw=1.6, label="2-component GMM (EM)")
plt.axvline(t_doc, color="k", ls="--", label=f"Otsu t={t_doc}")
plt.legend(fontsize=8); plt.title("Thresholding as mixture modelling")
plt.tight_layout(); plt.show()

## 9.5 Bayesian pixel classification: the prior earns its keep

With the mixture fitted, classifying a pixel is Bayes' rule:

$$P(\text{ink} \mid r) = \frac{p(r \mid \text{ink})\,P(\text{ink})}
{p(r\mid \text{ink})\,P(\text{ink}) + p(r\mid \text{paper})\,P(\text{paper})}$$

The **MAP decision** picks the larger posterior, which shifts the threshold away from where the two likelihoods
cross whenever the priors are unequal. On documents the prior is strongly asymmetric — typically 5–15% ink.

How much that shift matters is entirely a question of **class overlap**, and the experiment below measures it
on two scans. On a high-contrast scan the two Gaussians barely overlap, the posterior flips almost regardless
of the prior, and Otsu — which implicitly assumes balanced classes — is already close to optimal. On a faded,
blurred scan the classes overlap heavily and the same prior sweep moves the boundary across dozens of intensity
levels. This is the general behaviour of any Bayesian classifier: **the prior matters in proportion to how much
the likelihoods overlap**, which is also why class-imbalance corrections help far more on hard problems than
on easy ones.

In [ ]:
def map_threshold(mu_g, sd_g, prior_ink, grid=np.arange(256)):
    # MAP rule: label "ink" while the darker component has the larger posterior.
    # The boundary is the last intensity of the initial run where ink wins.
    dark, bright = (0, 1) if mu_g[0] < mu_g[1] else (1, 0)
    post_ink = prior_ink * stats.norm.pdf(grid, mu_g[dark], sd_g[dark])
    post_paper = (1 - prior_ink) * stats.norm.pdf(grid, mu_g[bright], sd_g[bright])
    win = post_ink > post_paper
    boundary = int(np.argmax(~win) - 1) if (~win).any() else 255
    return boundary, post_ink, post_paper

# Ground truth: the same layout rendered without blur or sensor noise
TRUTH = make_document(ink_sd=0.01, paper_sd=0.01, blur=0.01) < 140
print(f"true ink fraction: {100*TRUTH.mean():.2f}%\n")

print(f"{'prior P(ink)':>14}{'boundary':>10}{'ink pixels (%)':>16}{'error vs truth (%)':>21}")
print("-" * 61)
for prior in [0.02, 0.05, 0.10, 0.20, 0.50, 0.80]:
    b, _, _ = map_threshold(mu_g, sd_g, prior)
    pred = DOC <= b
    print(f"{prior:>14.2f}{b:>10}{100*pred.mean():>16.2f}{100*(pred != TRUTH).mean():>21.2f}")

b_emp, pi, pp = map_threshold(mu_g, sd_g, float(min(w)))
print(f"\nGMM-estimated ink prior {min(w):.4f} -> MAP boundary {b_emp}, "
      f"error {100*((DOC <= b_emp) != TRUTH).mean():.2f}%")
print(f"Otsu (equal-weight assumption)  -> boundary {t_doc}, "
      f"error {100*((DOC <= t_doc) != TRUTH).mean():.2f}%")

# How much the prior matters depends entirely on class overlap
FADED = make_document(ink=140, ink_sd=34, paper=205, paper_sd=18, blur=1.4)
w_f, mu_f, sd_f = fit_gmm_1d(FADED)
print(f"\nFaded low-contrast scan -- GMM means {np.round(mu_f,1)}, sds {np.round(sd_f,1)}")
print(f"{'prior P(ink)':>14}{'boundary (high contrast)':>26}{'boundary (faded)':>19}")
print("-" * 59)
for prior in [0.02, 0.10, 0.50, 0.80]:
    print(f"{prior:>14.2f}{map_threshold(mu_g, sd_g, prior)[0]:>26}"
          f"{map_threshold(mu_f, sd_f, prior)[0]:>19}")
print("\nWell-separated classes make the prior nearly irrelevant; overlapping classes make it decisive.")
print("That is the general rule for any Bayesian classifier, not only for pixels.")

grid = np.arange(256)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(grid, pi, label="P(ink) x p(r|ink)")
ax[0].plot(grid, pp, label="P(paper) x p(r|paper)")
ax[0].axvline(b_emp, color="k", ls="--", label=f"MAP boundary {b_emp}")
ax[0].set_yscale("log"); ax[0].legend(fontsize=8); ax[0].set_title("Posterior comparison (log scale)")
post = pi / (pi + pp + 1e-300)
ax[1].plot(grid, post); ax[1].axhline(0.5, color="r", ls=":")
ax[1].set_xlabel("intensity r"); ax[1].set_ylabel("P(ink | r)")
ax[1].set_title("Posterior probability of ink")
plt.tight_layout(); plt.show()

## 9.6 Entropy as a compression bound

Shannon's source coding theorem: any lossless code that encodes pixels **independently** needs at least $H(p)$
bits per pixel on average, where $p$ is the intensity distribution. That gives a target to measure real
compressors against — and the comparison is informative, because PNG routinely beats the pixel-wise entropy.

It does so by not coding pixels independently: it predicts each pixel from its neighbours and codes the
*residual*, whose distribution is far more concentrated (and roughly Laplacian, Section 4). This is the same
principle as an n-gram language model beating a unigram one: conditioning on context lowers entropy.

In [ ]:
def png_bits_per_pixel(img):
    ok, buf = cv2.imencode(".png", img, [cv2.IMWRITE_PNG_COMPRESSION, 9])
    return 8 * len(buf) / img.size

CLEAN_DOC = make_document(ink_sd=0.01, paper_sd=0.01, blur=0.7)   # same layout, no sensor noise

print(f"{'image':<24}{'H(pixels)':>12}{'H(residual)':>14}{'PNG bpp':>10}")
print("-" * 60)
for img, name in [(CAMERA, "camera"), (CLEAN_DOC, "document (noise-free)"), (DOC, "document"),
                  (noisy["gaussian sigma=12"], "document + noise")]:
    resid = np.diff(img.astype(int), axis=1).ravel()          # horizontal prediction residual
    p_res = np.bincount(resid - resid.min()).astype(float)
    p_res /= p_res.sum()
    print(f"{name:<24}{entropy(hist_pmf(img)):>12.4f}{entropy(p_res):>14.4f}{png_bits_per_pixel(img):>10.4f}")

print("\nThree readings:")
print("1. Conditioning pays. Residual entropy sits below pixel entropy, and PNG -- which predicts each")
print("   pixel from its neighbours -- comes in below the pixel-wise entropy bound on every image here.")
print("2. The bound is not actually violated: H(p) bounds SYMBOL-WISE codes. PNG conditions on context,")
print("   so the quantity that bounds it is the conditional entropy, which is much lower.")
print("3. Noise is what actually costs money. The same document layout goes from about 1.7 to about")
print("   6.2 bits/pixel once sensor noise is present -- roughly a 3.6x larger file for zero extra")
print("   information. Random values cannot be predicted from neighbours, so context has nothing to")
print("   exploit. Denoising before archiving shrinks a scanned archive more than switching codec does.")

---
# 10. The Shared Mathematics

Every technique in the two case studies is one of five operations on a distribution.

| Operation | LLM text completion | Digital image processing |
|---|---|---|
| **Estimate** a distribution | count n-grams, smooth with $\alpha$ (MAP) | build a histogram (MLE); fit a GMM with EM |
| **Reshape** it | temperature $T$ on the logits | histogram equalization / specification via the CDF |
| **Truncate** it | top-k, top-p (nucleus) | clipping, trimmed filters, outlier rejection |
| **Sample / decide** from it | inverse-CDF token sampling; greedy = argmax | MAP pixel labelling; Otsu threshold |
| **Measure** it | entropy, cross-entropy loss, perplexity, KL | histogram entropy, compression bound, KL between histograms |

Three transferable conclusions:

1. **Zero probability is a strong claim.** It broke n-gram perplexity in Section 8.3 and it is why Gaussian
   models fail on impulse noise in Section 9.3. Smoothing and robust estimators are the same remedy applied in
   two fields.
2. **Conditioning reduces entropy.** A 4-gram beats a 2-gram; a residual code beats a pixel code. Whenever a
   number looks too high, ask what you have not conditioned on yet.
3. **Choose the metric to match the decision.** Entropy, KL, and perplexity measure the distribution;
   accuracy, CER, and PSNR measure the outcome. They disagree often enough that reporting only one is how
   projects end up optimizing the wrong thing.

---
# 11. Exercises

Add new cells below. Each answer needs code, numbers, and a short interpretation.

**1 — Distribution identification (15).** Write `identify_noise(patch)` that returns one of
`"gaussian"`, `"poisson"`, `"impulse"`, `"uniform"` using only the flat patch: compare MLE and MAD scale
estimates, test the mean–variance relationship across brightness levels, and check for mass at 0 and 255.
Validate it on the four `noisy` images from Section 9.3 and report which one it gets wrong and why.

**2 — Temperature calibration (15).** For the n-gram model, find the temperature $T^\*$ that **minimizes test
perplexity**. Note that perplexity is computed from the temperature-scaled distribution. Is $T^\* = 1$? Explain
what it means if the optimum is below or above 1 (this is exactly the calibration problem in deployed LLMs).

**3 — Top-p versus top-k, quantified (20).** Over all decoding steps of the test text, compute the mean KL
divergence between the truncated and the original distribution, and the mean nucleus size, for
$k \in \{1,3,5,10\}$ and $p \in \{0.7,0.8,0.9,0.95\}$. Plot mean KL against mean support size and identify which
settings are Pareto-dominated.

**4 — Histogram specification (20).** Generalize `equalize` into `match_histogram(src, reference)` that maps
`src` so its histogram approximates that of `reference`, using $F_{\text{ref}}^{-1}(F_{\text{src}}(r))$. Apply
it to make the `camera` image match the document histogram, and report the KL divergence between the achieved
and the target histogram. Explain why it cannot reach zero.

**5 — Bayes with a spatial prior (20).** Replace the global ink prior in Section 9.5 with a **local** prior
estimated from a large-window mean, so $P(\text{ink})$ varies by position. Compare the resulting binarization
against Otsu on a version of `DOC` corrupted by uneven illumination (multiply by a smooth gradient). Report the
per-band error rate against the clean binarization as ground truth.

**6 — Reflection (10).** Sections 8.3 and 9.6 both show entropy dropping when context is added. State the shared
theorem, and give one example from your own field where an unconditional estimate is being used where a
conditional one is available.

---

### References

- Bishop, C. M. *Pattern Recognition and Machine Learning*, Springer, 2006 — Chapter 2 (probability
  distributions) and Chapter 9 (mixture models and EM).
- Cover, T. M. & Thomas, J. A. *Elements of Information Theory*, 2nd ed., Wiley, 2006 — entropy, KL divergence,
  source coding.
- Gonzalez, R. C. & Woods, R. E. *Digital Image Processing*, 4th ed., Pearson, 2018 — Chapter 3 (histogram
  processing), Chapter 5 (noise models), Chapter 10 (thresholding, Otsu).
- Holtzman, A. et al. "The Curious Case of Neural Text Degeneration", *ICLR*, 2020 — the nucleus (top-p)
  sampling paper, and the source of the greedy-degeneration argument in Section 8.4.
- Jurafsky, D. & Martin, J. H. *Speech and Language Processing*, 3rd ed. draft — n-gram models, smoothing,
  perplexity.

*Note: these references are written from memory without access to a bibliographic database, so please verify
page numbers, editions, and years before citing them in a syllabus or paper.*